In [1]:
!pip install spacy scikit-learn pandas -q
!python -m spacy download en_core_web_sm -q

import pandas as pd
import re
import hashlib
import spacy

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

nlp = spacy.load("en_core_web_sm")

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
df = pd.read_csv('edufeed_clean.csv')

df = df[df['comments'].notna() & (df['comments'] != '')]
df = df[df['sentiment_label'].notna()]
df.reset_index(drop=True, inplace=True)

In [3]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df['comments'] = df['comments'].apply(clean_text)

In [4]:
def generate_hash(text):
    return hashlib.sha256(text.encode()).hexdigest()

In [5]:
X = df['comments']
y = df['sentiment_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(
    max_features=7000,
    ngram_range=(1,3),
    min_df=2
)

X_train_vec = tfidf.fit_transform(X_train)

lr_model = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    C=2
)

lr_model.fit(X_train_vec, y_train)

LogisticRegression(C=2, class_weight='balanced', max_iter=2000)

In [6]:
def generate_hash(text):
    return hashlib.sha256(text.encode()).hexdigest()

In [7]:
def mask_text(text, name, roll, email):
    text = str(text)

    text = re.sub(r'[\w\.-]+@[\w\.-]+\.\w+', '[EMAIL]', text)

    text = text.replace(name, '[STUDENT]')
    text = text.replace(roll, '[ROLL]')
    text = text.replace(email, '[EMAIL]')

    text = re.sub(r'\b\d+\b', '[ROLL]', text)

    return text

In [8]:
def feedback_system():

    name = input("Enter your Name: ")
    roll = input("Enter your Roll No: ")
    email = input("Enter your Email: ")
    professor = input("Enter Professor Name: ")
    rating_input = float(input("Enter Rating (1-5): "))
    review = input("Enter your Review: ")

    student_id = generate_hash(name + roll + email)

    full_text = f"{name} {roll} {email} {review}"
    masked_review = mask_text(full_text, name, roll, email)

    cleaned_review = clean_text(masked_review)

    text_vec = tfidf.transform([cleaned_review])
    sentiment = lr_model.predict(text_vec)[0]

    print("\n----- OUTPUT -----")
    print("Review:", masked_review)
    print("Rating:", rating_input)
    print("Sentiment:", sentiment)

    return masked_review, rating_input, sentiment

In [9]:
feedback_system()


----- OUTPUT -----
Review: [STUDENT] [ROLL] [EMAIL] it was okay okay not bad not good
Rating: 4.0
Sentiment: negative


('[STUDENT] [ROLL] [EMAIL] it was okay okay not bad not good', 4.0, 'negative')

In [10]:
from sklearn.metrics import classification_report, f1_score
import matplotlib.pyplot as plt

# 1. THE AUDIT REPORT
X_test_vec = tfidf.transform(X_test)
y_pred = lr_model.predict(X_test_vec)

print("### 1. HYPERPARAMETER AUDIT: CLASSIFICATION REPORT ###")
print(f"Settings: C=2, max_iter=2000, class_weight='balanced', features=7000")
report = classification_report(y_test, y_pred)
print(report)

### 1. HYPERPARAMETER AUDIT: CLASSIFICATION REPORT ###
Settings: C=2, max_iter=2000, class_weight='balanced', features=7000
              precision    recall  f1-score   support

    negative       0.69      0.78      0.73      1134
     neutral       0.26      0.39      0.31       501
    positive       0.89      0.73      0.81      2360

    accuracy                           0.70      3995
   macro avg       0.61      0.64      0.62      3995
weighted avg       0.75      0.70      0.72      3995



In [11]:
# 2. NEUTRAL CLASS F1-SCORE AUDIT
neutral_label = 'Neutral' if 'Neutral' in y.unique() else 'neutral'
f1_neutral = f1_score(y_test, y_pred, labels=[neutral_label], average='weighted')

print(f"### 2. METRIC FOCUS: {neutral_label.upper()} CLASS ###")
print(f"F1-Score for '{neutral_label}': {f1_neutral:.4f}")
if f1_neutral < 0.6:
    print("AUDIT NOTE: Neutral class performance is low. Trigrams may be introducing noise.\n")


### 2. METRIC FOCUS: NEUTRAL CLASS ###
F1-Score for 'neutral': 0.3091
AUDIT NOTE: Neutral class performance is low. Trigrams may be introducing noise.



In [12]:

# 3. N-GRAM RANGE COMPARISON (BASELINE (1,2) vs. CURRENT (1,3))
print("### 3. N-GRAM RANGE AUDIT ###")
# Testing baseline (1,2) to see if trigrams (1,3) actually improved performance
tfidf_baseline = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=2)
X_train_base = tfidf_baseline.fit_transform(X_train)
X_test_base = tfidf_baseline.transform(X_test)

lr_base = LogisticRegression(max_iter=2000, class_weight='balanced', C=1)
lr_base.fit(X_train_base, y_train)
y_pred_base = lr_base.predict(X_test_base)

base_f1 = f1_score(y_test, y_pred_base, average='weighted')
current_f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Baseline (N-gram 1-2, 5000 features) Weighted F1: {base_f1:.4f}")
print(f"Current  (N-gram 1-3, 7000 features) Weighted F1: {current_f1:.4f}")

if current_f1 > base_f1:
    print("CONCLUSION: The increase to Trigrams and 7000 features improved generalization.")
else:
    print("CONCLUSION: Potential Overfitting. Baseline (1,2) performed better or equal.")

### 3. N-GRAM RANGE AUDIT ###
Baseline (N-gram 1-2, 5000 features) Weighted F1: 0.7266
Current  (N-gram 1-3, 7000 features) Weighted F1: 0.7221
CONCLUSION: Potential Overfitting. Baseline (1,2) performed better or equal.


In [13]:
# 4. FEATURE LIMIT CHECK (OVERFITTING)
train_acc = lr_model.score(X_train_vec, y_train)
test_acc = lr_model.score(X_test_vec, y_test)
print(f"\n### 4. OVERFITTING CHECK ###")
print(f"Training Accuracy: {train_acc:.4f}")
print(f"Testing Accuracy:  {test_acc:.4f}")
print(f"Generalization Gap: {train_acc - test_acc:.4f}")


### 4. OVERFITTING CHECK ###
Training Accuracy: 0.8514
Testing Accuracy:  0.7041
Generalization Gap: 0.1472
